# Tiny Qwen-style LM on WikiText-2 with AdamW vs Muon and Intra-layer Matrix Diagnostics

This notebook trains a very small Qwen-style causal language model from scratch on WikiText-2 using a simple word-level tokenizer.

The goal is educational: while the network trains, we inspect how individual matrices evolve inside layers. We track singular values, Frobenius norm, spectral norm, stable rank, top-k energy, row/column norm statistics, update sizes, and subspace alignment.

The notebook compares **AdamW** and **Muon** to show that optimizer choice changes the dynamics of learned matrices, even when the model and data are the same.

## 0. Install dependencies

Run this cell if needed. In many Colab/Kaggle/local notebook environments, `torch`, `datasets`, and `matplotlib` may already be installed.

In [ ]:
# Uncomment if your environment does not already have these packages.
# %pip install -q torch datasets matplotlib pandas tqdm

## 1. Imports and global configuration

Edit the `CONFIG` dictionary to change model size, optimizer settings, training length, logging interval, and matrix diagnostic options.

In [ ]:
import math
import random
import re
import time
from dataclasses import dataclass
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from tqdm.auto import tqdm

# Reproducibility.
def set_seed(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

CONFIG = {
    # Data / tokenizer.
    "dataset_name": "Salesforce/wikitext",
    "dataset_config": "wikitext-2-raw-v1",
    "vocab_size": 8000,
    "seq_len": 64,
    "train_max_lines": 6000,      # lower for CPU demos; increase for better curves
    "val_max_lines": 1000,

    # Tiny Qwen-style model.
    "d_model": 128,
    "n_layers": 2,
    "n_heads": 4,
    "d_ff": 384,
    "dropout": 0.0,

    # Training.
    "batch_size": 32,
    "max_steps": 600,
    "eval_interval": 50,
    "diag_interval": 50,
    "grad_clip": 1.0,

    # AdamW hyperparameters.
    "adamw_lr": 3e-4,
    "adamw_betas": (0.9, 0.95),
    "adamw_weight_decay": 0.1,

    # Muon hyperparameters.
    "muon_lr": 0.02,
    "muon_momentum": 0.95,
    "muon_weight_decay": 0.0,
    "muon_ns_steps": 5,

    # Matrix diagnostics.
    "top_k": 8,
    "matrix_name_patterns": [
        "attn.q_proj.weight",
        "attn.k_proj.weight",
        "attn.v_proj.weight",
        "attn.o_proj.weight",
        "mlp.gate_proj.weight",
        "mlp.up_proj.weight",
        "mlp.down_proj.weight",   # FFN W_out in this implementation
    ],
}

## 2. Load WikiText-2 and build a simple word-level tokenizer

This intentionally uses a simple tokenizer so that the notebook stays transparent. It is not meant to match Qwen's production tokenizer.

In [ ]:
raw = load_dataset(CONFIG["dataset_name"], CONFIG["dataset_config"])
print(raw)

_token_re = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def simple_word_tokenize(text: str):
    # Lowercase word-level tokenizer with punctuation as separate tokens.
    return _token_re.findall(text.lower())

SPECIALS = ["<pad>", "<unk>", "<bos>", "<eos>"]
PAD, UNK, BOS, EOS = range(4)

def take_nonempty_lines(split, max_lines):
    lines = []
    for row in raw[split]:
        text = row["text"].strip()
        if text:
            lines.append(text)
        if len(lines) >= max_lines:
            break
    return lines

train_lines = take_nonempty_lines("train", CONFIG["train_max_lines"])
val_lines = take_nonempty_lines("validation", CONFIG["val_max_lines"])

counter = Counter()
for line in train_lines:
    counter.update(simple_word_tokenize(line))

most_common = [tok for tok, _ in counter.most_common(CONFIG["vocab_size"] - len(SPECIALS))]
itos = SPECIALS + most_common
stoi = {tok: i for i, tok in enumerate(itos)}

print("vocab size:", len(itos))
print("example tokens:", itos[:30])

def encode_line(text: str):
    ids = [BOS]
    ids += [stoi.get(tok, UNK) for tok in simple_word_tokenize(text)]
    ids += [EOS]
    return ids

def encode_lines(lines):
    ids = []
    for line in lines:
        ids.extend(encode_line(line))
    return torch.tensor(ids, dtype=torch.long)

train_ids = encode_lines(train_lines)
val_ids = encode_lines(val_lines)
print("train tokens:", len(train_ids), "val tokens:", len(val_ids))

## 3. Dataset: fixed-length causal LM blocks

Each sample returns `x = tokens[t:t+seq_len]` and `y = tokens[t+1:t+seq_len+1]`.

In [ ]:
class TokenBlockDataset(Dataset):
    def __init__(self, ids: torch.Tensor, seq_len: int):
        assert ids.ndim == 1
        self.ids = ids
        self.seq_len = seq_len

    def __len__(self):
        return max(0, len(self.ids) - self.seq_len - 1)

    def __getitem__(self, idx):
        x = self.ids[idx:idx+self.seq_len]
        y = self.ids[idx+1:idx+self.seq_len+1]
        return x, y

def make_loader(ids, shuffle=True):
    ds = TokenBlockDataset(ids, CONFIG["seq_len"])
    return DataLoader(ds, batch_size=CONFIG["batch_size"], shuffle=shuffle, drop_last=True)

train_loader = make_loader(train_ids, shuffle=True)
val_loader = make_loader(val_ids, shuffle=False)
print("train batches:", len(train_loader), "val batches:", len(val_loader))

## 4. Tiny Qwen-style model

This is a tiny educational approximation of the Qwen/Qwen2 block style:

- RMSNorm
- RoPE rotary position embeddings
- causal self-attention
- SwiGLU MLP with `gate_proj`, `up_proj`, and `down_proj`

In this notebook, `mlp.down_proj.weight` is the FFN output matrix, i.e. the matrix analogous to `W_out`.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        return self.weight * x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=4096, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, x):
        # x shape: [batch, heads, seq, head_dim]
        seq_len = x.size(-2)
        cos = self.cos[:seq_len].to(x.device)[None, None, :, :]
        sin = self.sin[:seq_len].to(x.device)[None, None, :, :]
        x1 = x[..., ::2]
        x2 = x[..., 1::2]
        out = torch.empty_like(x)
        out[..., ::2] = x1 * cos - x2 * sin
        out[..., 1::2] = x1 * sin + x2 * cos
        return out

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)
        self.rope = RotaryEmbedding(self.head_dim)
        self.dropout = dropout

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        q = self.rope(q)
        k = self.rope(k)
        # PyTorch applies causal masking internally when is_causal=True.
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(y)

class SwiGLUMLP(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class TinyQwenBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.0):
        super().__init__()
        self.input_layernorm = RMSNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout=dropout)
        self.post_attention_layernorm = RMSNorm(d_model)
        self.mlp = SwiGLUMLP(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.input_layernorm(x))
        x = x + self.mlp(self.post_attention_layernorm(x))
        return x

class TinyQwenLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, seq_len, dropout=0.0):
        super().__init__()
        self.embed_tokens = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            TinyQwenBlock(d_model, n_heads, d_ff, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.seq_len = seq_len
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        x = self.embed_tokens(idx)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

model = TinyQwenLM(
    vocab_size=len(itos),
    d_model=CONFIG["d_model"],
    n_layers=CONFIG["n_layers"],
    n_heads=CONFIG["n_heads"],
    d_ff=CONFIG["d_ff"],
    seq_len=CONFIG["seq_len"],
    dropout=CONFIG["dropout"],
)
print("parameters:", sum(p.numel() for p in model.parameters())/1e6, "M")
for name, p in model.named_parameters():
    if any(name.endswith(pattern) for pattern in CONFIG["matrix_name_patterns"]):
        print(name, tuple(p.shape))

## 5. Muon optimizer

Muon orthogonalizes 2D matrix updates using Newton-Schulz iterations. This simple implementation is for demonstration, not a production distributed optimizer.

For embeddings and 1D parameters, the training code below uses AdamW as a companion optimizer when `optimizer_name="muon"`.

In [ ]:
@torch.no_grad()
def zeropower_via_newton_schulz(G, steps=5, eps=1e-7):
    # Approximate the polar factor / orthogonalized update direction.
    # The coefficients below are commonly used in Muon-style implementations.
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.float()
    if X.ndim != 2:
        return G
    transpose = False
    if X.size(0) > X.size(1):
        X = X.T
        transpose = True
    X = X / (X.norm() + eps)
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * (A @ A)
        X = a * X + B @ X
    if transpose:
        X = X.T
    return X.to(dtype=G.dtype)

class SimpleMuon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, momentum=0.95, weight_decay=0.0, ns_steps=5):
        defaults = dict(lr=lr, momentum=momentum, weight_decay=weight_decay, ns_steps=ns_steps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr = group["lr"]
            momentum = group["momentum"]
            weight_decay = group["weight_decay"]
            ns_steps = group["ns_steps"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                if weight_decay != 0:
                    p.mul_(1 - lr * weight_decay)
                g = p.grad
                state = self.state[p]
                if "momentum_buffer" not in state:
                    state["momentum_buffer"] = torch.zeros_like(p)
                buf = state["momentum_buffer"]
                buf.mul_(momentum).add_(g)
                update = zeropower_via_newton_schulz(buf, steps=ns_steps)
                # Scale update to keep magnitude roughly comparable across aspect ratios.
                scale = max(1.0, math.sqrt(p.shape[0] / p.shape[1])) if p.ndim == 2 else 1.0
                p.add_(update, alpha=-lr * scale)
        return loss

def split_params_for_muon(model):
    # Muon is intended for hidden 2D matrices; use AdamW for embeddings, LM head, norms, and 1D params.
    muon_params, adam_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        use_muon = (p.ndim == 2) and ("embed" not in name) and ("lm_head" not in name)
        if use_muon:
            muon_params.append(p)
        else:
            adam_params.append(p)
    return muon_params, adam_params

def make_optimizers(model, optimizer_name: str):
    optimizer_name = optimizer_name.lower()
    if optimizer_name == "adamw":
        return [torch.optim.AdamW(
            model.parameters(),
            lr=CONFIG["adamw_lr"],
            betas=CONFIG["adamw_betas"],
            weight_decay=CONFIG["adamw_weight_decay"],
        )]
    if optimizer_name == "muon":
        muon_params, adam_params = split_params_for_muon(model)
        opts = []
        opts.append(SimpleMuon(
            muon_params,
            lr=CONFIG["muon_lr"],
            momentum=CONFIG["muon_momentum"],
            weight_decay=CONFIG["muon_weight_decay"],
            ns_steps=CONFIG["muon_ns_steps"],
        ))
        if adam_params:
            opts.append(torch.optim.AdamW(
                adam_params,
                lr=CONFIG["adamw_lr"],
                betas=CONFIG["adamw_betas"],
                weight_decay=CONFIG["adamw_weight_decay"],
            ))
        return opts
    raise ValueError(f"Unknown optimizer: {optimizer_name}")

## 6. Matrix diagnostics

These functions compute intra-layer quantities for each selected matrix:

- Frobenius norm
- spectral norm
- stable rank
- top-k spectral energy ratio
- row/column norm max-to-mean ratios
- relative update norm between checkpoints
- top singular vector alignment
- top-k subspace alignment
- gradient projection onto the top singular directions

In [ ]:
def should_track_matrix(name):
    return any(name.endswith(pattern) for pattern in CONFIG["matrix_name_patterns"])

@torch.no_grad()
def svd_diagnostics(W, top_k=8):
    Wf = W.detach().float().cpu()
    U, S, Vh = torch.linalg.svd(Wf, full_matrices=False)
    k = min(top_k, S.numel())
    fro = torch.linalg.norm(Wf, ord="fro").item()
    spectral = S[0].item() if S.numel() else float("nan")
    stable_rank = (fro ** 2) / (spectral ** 2 + 1e-12)
    topk_energy = (S[:k].pow(2).sum() / S.pow(2).sum()).item()
    row_norms = torch.linalg.norm(Wf, dim=1)
    col_norms = torch.linalg.norm(Wf, dim=0)
    return {
        "fro_norm": fro,
        "spectral_norm": spectral,
        "stable_rank": stable_rank,
        "topk_energy": topk_energy,
        "row_norm_max_over_mean": (row_norms.max() / (row_norms.mean() + 1e-12)).item(),
        "col_norm_max_over_mean": (col_norms.max() / (col_norms.mean() + 1e-12)).item(),
        "singular_values": S[:k].numpy().tolist(),
        "U_top": U[:, :k].clone(),
        "V_top": Vh[:k, :].T.clone(),
    }

@torch.no_grad()
def subspace_overlap(A, B):
    # Mean squared singular value of A^T B. 1.0 means identical subspaces.
    k = min(A.shape[1], B.shape[1])
    if k == 0:
        return float("nan")
    M = A[:, :k].T @ B[:, :k]
    s = torch.linalg.svdvals(M)
    return s.pow(2).mean().item()

@torch.no_grad()
def collect_matrix_diagnostics(model, step, optimizer_name, previous_state=None, top_k=8):
    rows = []
    new_state = {}
    for name, p in model.named_parameters():
        if not should_track_matrix(name):
            continue
        diag = svd_diagnostics(p, top_k=top_k)
        W_cpu = p.detach().float().cpu().clone()
        new_state[name] = {
            "W": W_cpu,
            "U_top": diag.pop("U_top"),
            "V_top": diag.pop("V_top"),
        }
        row = {"step": step, "optimizer": optimizer_name, "matrix": name, **diag}

        if previous_state is not None and name in previous_state:
            W_prev = previous_state[name]["W"]
            dW = W_cpu - W_prev
            row["rel_delta_fro"] = (torch.linalg.norm(dW, ord="fro") / (torch.linalg.norm(W_prev, ord="fro") + 1e-12)).item()
            row["rel_delta_spectral"] = (torch.linalg.svdvals(dW)[0] / (torch.linalg.svdvals(W_prev)[0] + 1e-12)).item()
            # Top singular vector sign is arbitrary, so use absolute cosine.
            row["top_v_cos"] = torch.abs(torch.dot(previous_state[name]["V_top"][:, 0], new_state[name]["V_top"][:, 0])).item()
            row["top_u_cos"] = torch.abs(torch.dot(previous_state[name]["U_top"][:, 0], new_state[name]["U_top"][:, 0])).item()
            row["V_topk_overlap"] = subspace_overlap(previous_state[name]["V_top"], new_state[name]["V_top"])
            row["U_topk_overlap"] = subspace_overlap(previous_state[name]["U_top"], new_state[name]["U_top"])
        else:
            row["rel_delta_fro"] = float("nan")
            row["rel_delta_spectral"] = float("nan")
            row["top_v_cos"] = float("nan")
            row["top_u_cos"] = float("nan")
            row["V_topk_overlap"] = float("nan")
            row["U_topk_overlap"] = float("nan")

        # Gradient projection onto the top singular rank-1 directions u_i v_i^T.
        if p.grad is not None:
            G = p.grad.detach().float().cpu()
            U = new_state[name]["U_top"]
            V = new_state[name]["V_top"]
            projections = []
            for i in range(min(top_k, U.shape[1], V.shape[1])):
                projections.append(float(U[:, i] @ G @ V[:, i]))
            row["grad_top_svd_projection_l2"] = float(torch.tensor(projections).norm())
            row["grad_fro_norm"] = float(torch.linalg.norm(G, ord="fro"))
        else:
            row["grad_top_svd_projection_l2"] = float("nan")
            row["grad_fro_norm"] = float("nan")
        rows.append(row)
    return rows, new_state

## 7. Training and evaluation loop

The function below trains one optimizer variant and returns two tables:

1. `loss_df`: train and validation loss over time
2. `diag_df`: matrix diagnostics over time

In [ ]:
@torch.no_grad()
def evaluate(model, loader, max_batches=20):
    model.eval()
    losses = []
    for i, (x, y) in enumerate(loader):
        if i >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return sum(losses) / max(1, len(losses))

def cycle_loader(loader):
    while True:
        for batch in loader:
            yield batch

def train_one_run(optimizer_name: str):
    set_seed(42)  # same initialization for fair comparison
    model = TinyQwenLM(
        vocab_size=len(itos),
        d_model=CONFIG["d_model"],
        n_layers=CONFIG["n_layers"],
        n_heads=CONFIG["n_heads"],
        d_ff=CONFIG["d_ff"],
        seq_len=CONFIG["seq_len"],
        dropout=CONFIG["dropout"],
    ).to(device)

    optimizers = make_optimizers(model, optimizer_name)
    loader_iter = cycle_loader(train_loader)

    loss_rows = []
    diag_rows_all = []
    prev_diag_state = None

    pbar = tqdm(range(CONFIG["max_steps"] + 1), desc=f"training {optimizer_name}")
    for step in pbar:
        if step % CONFIG["eval_interval"] == 0:
            val_loss = evaluate(model, val_loader)
            loss_rows.append({"step": step, "optimizer": optimizer_name, "split": "validation", "loss": val_loss})

        if step % CONFIG["diag_interval"] == 0:
            rows, prev_diag_state = collect_matrix_diagnostics(
                model, step, optimizer_name, previous_state=prev_diag_state, top_k=CONFIG["top_k"]
            )
            diag_rows_all.extend(rows)

        if step == CONFIG["max_steps"]:
            break

        x, y = next(loader_iter)
        x, y = x.to(device), y.to(device)
        for opt in optimizers:
            opt.zero_grad(set_to_none=True)
        _, loss = model(x, y)
        loss.backward()
        if CONFIG["grad_clip"] is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
        for opt in optimizers:
            opt.step()

        loss_rows.append({"step": step, "optimizer": optimizer_name, "split": "train", "loss": loss.item()})
        if step % 20 == 0:
            pbar.set_postfix(loss=loss.item())

    return model, pd.DataFrame(loss_rows), pd.DataFrame(diag_rows_all)

## 8. Run AdamW and Muon experiments

For a fast CPU demonstration, lower `max_steps`, `train_max_lines`, `d_model`, or `batch_size` in `CONFIG` and restart the notebook.

In [ ]:
results = {}
for optimizer_name in ["adamw", "muon"]:
    model_out, loss_df, diag_df = train_one_run(optimizer_name)
    results[optimizer_name] = {"model": model_out, "loss_df": loss_df, "diag_df": diag_df}

loss_all = pd.concat([results[k]["loss_df"] for k in results], ignore_index=True)
diag_all = pd.concat([results[k]["diag_df"] for k in results], ignore_index=True)

loss_all.head(), diag_all.head()

## 9. Plot train and validation loss curves

In [ ]:
def plot_loss_curves(loss_all):
    for split in ["train", "validation"]:
        plt.figure(figsize=(8, 4))
        for opt_name, df in loss_all[loss_all["split"] == split].groupby("optimizer"):
            # For train loss, smooth lightly to make the plot readable.
            d = df.sort_values("step").copy()
            if split == "train":
                d["loss_plot"] = d["loss"].rolling(20, min_periods=1).mean()
            else:
                d["loss_plot"] = d["loss"]
            plt.plot(d["step"], d["loss_plot"], label=opt_name)
        plt.title(f"{split.capitalize()} loss")
        plt.xlabel("step")
        plt.ylabel("cross-entropy loss")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

plot_loss_curves(loss_all)

## 10. Plot spectral dynamics for one FFN output matrix

Here we focus on `layers.0.mlp.down_proj.weight`, which is the first layer's FFN output matrix.

In [ ]:
TARGET_MATRIX = "layers.0.mlp.down_proj.weight"

metrics_to_plot = [
    "fro_norm",
    "spectral_norm",
    "stable_rank",
    "topk_energy",
    "rel_delta_fro",
    "V_topk_overlap",
    "row_norm_max_over_mean",
    "col_norm_max_over_mean",
]

for metric in metrics_to_plot:
    plt.figure(figsize=(8, 4))
    for opt_name, df in diag_all[diag_all["matrix"] == TARGET_MATRIX].groupby("optimizer"):
        d = df.sort_values("step")
        plt.plot(d["step"], d[metric], marker="o", label=opt_name)
    plt.title(f"{TARGET_MATRIX}\n{metric}")
    plt.xlabel("step")
    plt.ylabel(metric)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

## 11. Compare all tracked matrices at final checkpoint

This table helps identify which matrices changed most and which became more spectrally concentrated.

In [ ]:
final_step = diag_all["step"].max()
summary = diag_all[diag_all["step"] == final_step].copy()
cols = [
    "optimizer", "matrix", "fro_norm", "spectral_norm", "stable_rank", "topk_energy",
    "rel_delta_fro", "V_topk_overlap", "row_norm_max_over_mean", "col_norm_max_over_mean",
]
summary[cols].sort_values(["optimizer", "matrix"])

## 12. Singular value curves for selected matrices

This visualizes the spectrum directly. A steeper curve means stronger concentration into a few dominant directions.

In [ ]:
def plot_singular_values(diag_all, matrix_name, step=None):
    if step is None:
        step = diag_all["step"].max()
    plt.figure(figsize=(8, 4))
    for opt_name, df in diag_all[(diag_all["matrix"] == matrix_name) & (diag_all["step"] == step)].groupby("optimizer"):
        s = df.iloc[0]["singular_values"]
        plt.plot(range(1, len(s)+1), s, marker="o", label=opt_name)
    plt.title(f"Top singular values at step {step}\n{matrix_name}")
    plt.xlabel("rank index")
    plt.ylabel("singular value")
    plt.yscale("log")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

plot_singular_values(diag_all, "layers.0.mlp.down_proj.weight")
plot_singular_values(diag_all, "layers.1.mlp.down_proj.weight" if CONFIG["n_layers"] > 1 else "layers.0.mlp.down_proj.weight")

## 13. Gradient projection onto dominant singular directions

This checks whether the current gradient points into the already-dominant matrix directions. A large value means the optimizer is mostly reinforcing/modifying existing dominant spectral directions; a small value relative to total gradient norm can indicate learning in other directions.

In [ ]:
for matrix_name in ["layers.0.mlp.down_proj.weight", "layers.0.mlp.up_proj.weight", "layers.0.attn.o_proj.weight"]:
    plt.figure(figsize=(8, 4))
    for opt_name, df in diag_all[diag_all["matrix"] == matrix_name].groupby("optimizer"):
        d = df.sort_values("step").copy()
        ratio = d["grad_top_svd_projection_l2"] / (d["grad_fro_norm"] + 1e-12)
        plt.plot(d["step"], ratio, marker="o", label=opt_name)
    plt.title(f"Gradient projection ratio\n{matrix_name}")
    plt.xlabel("step")
    plt.ylabel("||projection onto top SVD dirs|| / ||grad||")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

## 14. Interpretation guide

Use the plots and tables as follows:

- **Frobenius norm**: total matrix energy.
- **Spectral norm**: strength of the most amplified direction.
- **Stable rank**: soft effective rank. Low stable rank means energy is concentrated in few directions.
- **Top-k energy**: fraction of matrix energy in the first `k` singular values.
- **Relative delta Frobenius**: how much the matrix changed since the previous diagnostic checkpoint.
- **Top-k subspace overlap**: whether the dominant input/output subspaces remain stable. Near 1 means stable; lower values mean rotation.
- **Row/column max-over-mean norm ratios**: possible outlier neurons or outlier input directions.
- **Gradient projection ratio**: whether gradients align with dominant spectral directions.

Expected qualitative difference:

- AdamW usually produces update dynamics tied to coordinate-wise adaptive scaling.
- Muon orthogonalizes 2D hidden-layer updates, so its matrix spectra and subspace rotations can look quite different.

Do not over-interpret one tiny run. Repeat with different seeds, longer training, and larger models if you want robust conclusions.

## 15. Optional: save diagnostics to CSV

In [ ]:
loss_all.to_csv("loss_curves_adamw_vs_muon.csv", index=False)
diag_all.to_csv("matrix_diagnostics_adamw_vs_muon.csv", index=False)
print("saved CSV files")